# Can line counts recover an image?
### A finite Radon transform built from incidences

A picture becomes an arrangement of measurements. Those measurements move a second
arrangement until the original picture emerges. No image-inversion routine is used:
the construction is **relation → count → bind → sum**.

We begin with a failure of row/column counts, introduce modular lines, and derive an
exact reconstruction formula. The small examples use a prime-sized grid with binary
integer values. The [lesson notes](../docs/lessons/05_finite_radon.md) discuss weighted
images and the construction's limits.

Use the **Kaleion** kernel and **Run All**. [Setup and exports](README.md).

In [ ]:
from pathlib import Path
from math import isqrt
import json
import sys

import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from IPython.display import Markdown, display
from kaleion import Collection, Product, F, Inspection, Motion, Workspace, choose, param, vector
from kaleion.viewers.plotly import animation_figure

pio.renderers.default = "plotly_mimetype+notebook"
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "pyproject.toml").exists() and (p / "src/kaleion").is_dir())
if str(ROOT / "notebooks") not in sys.path:
    sys.path.insert(0, str(ROOT / "notebooks"))
from snapshot_views import compare_keyed_values, keyed_values, rectangular_values

OUTPUT = ROOT / "build" / "notebooks" / "finite-radon"
OUTPUT.mkdir(parents=True, exist_ok=True)
PARAMETERS = {"p": 5, "image_x": 2}
P = PARAMETERS["p"]
assert isinstance(P, int) and not isinstance(P, bool) and P >= 2
assert all(P % d for d in range(2, isqrt(P) + 1)), "Use a prime modulus."
assert P**3 * (P + 1) <= 10000, "Use p=2,3,5,7 for this fully displayed pair domain."
assert 0 <= PARAMETERS["image_x"] < P
p, image_x = param("p"), param("image_x")
BG, INK, TEAL, GOLD = "#101b2b", "#e8eef7", "#49c5b6", "#f0bc63"

## 1 · Two projections lose information

Consider a diagonal and an opposite diagonal in a $2\times2$ grid. Each row contains
one selected point; each column contains one selected point. The measurements agree
even though the selected subsets differ. Provenance can tell us which picture was
measured, but it does not make the numerical measurements alone sufficient to recover it.

In [ ]:
small_grid = Collection.grid(2, 2, values=1).arrange(F.i, F.j)
diagonal = small_grid.where(F.i == F.j)
opposite = small_grid.where(F.i + F.j == 1)
ambiguity = Workspace({"diagonal": diagonal, "opposite": opposite,
                       "rows_A": diagonal.count(by=F.i), "rows_B": opposite.count(by=F.i),
                       "cols_A": diagonal.count(by=F.j), "cols_B": opposite.count(by=F.j)})
ar = ambiguity.state.results
assert not ambiguity.state.errors
assert ar["rows_A"].values.tolist() == ar["rows_B"].values.tolist() == [1, 1]
assert ar["cols_A"].values.tolist() == ar["cols_B"].values.tolist() == [1, 1]
assert not np.array_equal(ar["diagonal"].mask, ar["opposite"].mask)

ambiguity_plot = make_subplots(rows=1, cols=2, subplot_titles=["Diagonal", "Opposite diagonal"])
for col, name in enumerate(("diagonal", "opposite"), 1):
    mask = ar[name].mask.reshape(2, 2).T.astype(int).tolist()
    ambiguity_plot.add_trace(go.Heatmap(z=mask, x=[0, 1], y=[0, 1], zmin=0, zmax=1,
                                        colorscale=[[0, BG], [1, TEAL]], showscale=False,
                                        xgap=6, ygap=6, text=[[str(v) for v in row] for row in mask],
                                        texttemplate="%{text}"), row=1, col=col)
ambiguity_plot.update_layout(template="plotly_dark", paper_bgcolor=BG, plot_bgcolor=BG,
                             title="Different pictures · identical row and column counts [1, 1]", height=370)
ambiguity_plot.update_xaxes(dtick=1, title_text="x")
ambiguity_plot.update_yaxes(dtick=1, title_text="y")
ambiguity_plot.show()

## 2 · Add modular directions

Use the $p^2$ points $(x,y)$ with $0\le x,y<p$, for prime $p$. A finite line is

\[
\ell_{m,t}: y-mx\equiv t\pmod p\quad(0\le m<p),
\qquad \ell_{p,t}:x=t\quad\text{(vertical)}.
\]

There are $p+1$ direction families, each with $p$ lines. These are finite-field
incidences: the points of a line can wrap across the drawing, so we highlight them
without inventing Euclidean connecting segments.

`pixels` and `lines` declare the two key domains. The pair arrangement declares
every possible point/line incidence. `Product(point=pixels, line=lines)` names
the source roles; `.read` copies the required source fields. The role indices are
current slots, while `(u,v)` and `(m,t)` retain the semantic keys. With $p=5$,
this is 750 pairs; the named recipe does not change that dense cost.

In [ ]:
pixels = Collection.grid(p, p, values=0).annotate(u=F.i, v=F.j)
lines = Collection.grid(p + 1, p, values=0).annotate(m=F.i, t=F.j)
image = pixels.with_values(choose(
    (F.u == image_x) | (F.v == p // 2) | ((F.u == 0) & (F.v == 0)), 1, 0))

point_line = Product(point=pixels, line=lines)
pairs = point_line.domain.annotate(
    u=point_line.read("point", F.u), v=point_line.read("point", F.v),
    m=point_line.read("line", F.m), t=point_line.read("line", F.t))
on_line = (((F.m < p) & ((F.v - F.m * F.u - F.t) % p == 0))
           | ((F.m == p) & (F.u == F.t)))
incidence = pairs.where(on_line)

sampling = pairs.annotate(weight=image.bind(on=(F.u, F.v), key=(F.u, F.v)))
illuminated = sampling.where(on_line & (F.weight == 1))
counts = illuminated.count(by=(F.m, F.t)).arrange(F.m, F.t)
# For this binary image, weighted sums and incidence counts agree.
weighted_sums = sampling.where(on_line).sum(by=(F.m, F.t), value=F.weight)

## 3 · Measurements become inputs to reconstruction

Let $S_\ell$ be the count on line $\ell$. At each pixel, add the measurements of
all incident lines:

\[
B(P)=\sum_{\ell\ni P}S_\ell.
\]

The total image weight $T$ is available from the measurements: sum all lines in any
one parallel family. We choose $m=0$. Reconstruction uses **geometry and measured
values**; it never reads an original image value to fill a reconstructed pixel.
The source image is consulted afterward to check the result.

In [ ]:
measured_value = counts.bind(on=(F.m, F.t), key=(F.m, F.t))
backprojection = incidence.sum(by=(F.u, F.v), value=measured_value)
total_from_measurements = counts.where(F.m == 0).sum().scalar()
numerator = backprojection.with_values(F.value - total_from_measurements)
remainders = numerator.with_values(F.value % p)
recovered = numerator.with_values(F.value // p)
residual = recovered.with_values(F.value - image.bind(on=(F.u, F.v), key=(F.u, F.v)))

ROOTS = {"image": image.arrange(F.u, F.v), "pairs": pairs, "sampling": sampling,
         "incidence": incidence, "counts": counts, "weighted_sums": weighted_sums,
         "backprojection": backprojection, "numerator": numerator, "remainders": remainders,
         "recovered": recovered.arrange(F.u, F.v), "residual": residual.arrange(F.u, F.v),
         "line_population": incidence.count(by=(F.m, F.t)),
         "directions_per_pixel": incidence.count(by=(F.u, F.v)),
         "family_totals": counts.sum(by=F.m)}
workspace = Workspace(ROOTS, PARAMETERS)
assert not workspace.state.errors, dict(workspace.state.errors)
state = workspace.state


def check_reconstruction(captured):
    assert not captured.errors, dict(captured.errors)
    r, size = captured.results, captured.parameters["p"]
    expected_domain = tuple((x, y) for x in range(size) for y in range(size))
    expected_keys = set(expected_domain)
    comparison = compare_keyed_values(r["recovered"], r["image"],
                                      left_keys=("u", "v"), domain=expected_domain)
    assert set(keyed_values(r["image"], keys=("u", "v"))) == set(keyed_values(r["recovered"], keys=("u", "v"))) == expected_keys
    assert set(r["image"].values) <= {0, 1}, "Use count for binary images; use weighted sum otherwise."
    assert set(r["line_population"].values) == {size}
    assert set(r["directions_per_pixel"].values) == {size + 1}
    assert keyed_values(r["counts"], keys=("m", "t")) == keyed_values(r["weighted_sums"], keys=("m", "t"))
    assert set(r["remainders"].values) == {0}, "Check exact division before accepting reconstruction."
    assert comparison.holds
    assert set(r["residual"].values) == {0}
    assert set(r["family_totals"].values) == {sum(r["image"].values)}
    return {"p": size, "pixels": size**2, "lines": size * (size + 1),
            "total_from_each_family": list(map(int, r["family_totals"].values)),
            "division_remainders": sorted(set(map(int, r["remainders"].values))),
            "exact_recovery": True, "comparison": comparison.to_dict()}

report = check_reconstruction(state)
print(json.dumps(report, indent=2))

In [ ]:
def heatmap(snapshot, names=("u", "v"), maximum=None):
    xs, ys, data = rectangular_values(snapshot, x=names[0], y=names[1])
    return go.Heatmap(x=xs, y=ys, z=data, zmin=0,
                      zmax=maximum if maximum is not None else max(1, max(map(max, data))),
                      colorscale=[[0, BG], [1, TEAL]], showscale=False, xgap=3, ygap=3,
                      text=[[str(v) for v in row] for row in data], texttemplate="%{text}",
                      hovertemplate="(%{x},%{y})<br>value=%{text}<extra></extra>")

def line_explorer(captured):
    r, size = captured.results, captured.parameters["p"]
    fig = make_subplots(rows=1, cols=2, subplot_titles=["Image · selected line outlined",
                                                      "Line counts S(m,t)"])
    fig.add_trace(heatmap(r["image"], maximum=1), row=1, col=1)
    fig.add_trace(go.Scatter(x=[], y=[], mode="markers", marker=dict(symbol="square-open", size=37,
                         color=GOLD, line=dict(width=3)), hoverinfo="skip", showlegend=False), row=1, col=1)
    fig.add_trace(heatmap(r["counts"], ("m", "t"), maximum=size), row=1, col=2)
    fig.add_trace(go.Scatter(x=[], y=[], mode="markers", marker=dict(symbol="square-open", size=37,
                         color=GOLD, line=dict(width=3)), hoverinfo="skip", showlegend=False), row=1, col=2)
    source = r["incidence"].source
    measured = keyed_values(r["counts"], keys=("m", "t"))
    frames, steps = [], []
    for m in range(size + 1):
        for t in range(size):
            selected = r["incidence"].mask & (source.fields["m"] == m) & (source.fields["t"] == t)
            name = f"{m}:{t}"
            formula = f"x = {t}" if m == size else f"y − {m}x ≡ {t} (mod {size})"
            title = f"{formula} · count {measured[m,t]}"
            frames.append(go.Frame(name=name, traces=[1, 3], data=[
                go.Scatter(x=list(map(int, source.fields["u"][selected])),
                           y=list(map(int, source.fields["v"][selected]))),
                go.Scatter(x=[m], y=[t])], layout=dict(title=dict(text=title))))
            steps.append(dict(label=name, method="animate", args=[[name], dict(mode="immediate",
                              frame=dict(duration=0, redraw=True), transition=dict(duration=0))]))
    fig.frames = frames
    for index, trace in zip((1, 3), frames[0].data):
        fig.data[index].x, fig.data[index].y = trace.x, trace.y
    fig.update_layout(template="plotly_dark", paper_bgcolor=BG, plot_bgcolor=BG, height=610,
                      title=frames[0].layout.title, margin=dict(t=110, b=180),
                      sliders=[dict(steps=steps, currentvalue=dict(prefix="Direction : intercept  "),
                                    x=.35, len=.65, y=-.21)],
                      updatemenus=[dict(type="buttons", direction="right", showactive=False,
                        y=-.23, x=0, xanchor="left", yanchor="top",
                        buttons=[dict(label="Play lines", method="animate", args=[None, dict(
                            frame=dict(duration=550, redraw=True), transition=dict(duration=0), fromcurrent=True)]),
                                 dict(label="Pause", method="animate", args=[[None], dict(mode="immediate",
                                    frame=dict(duration=0), transition=dict(duration=0))])])])
    fig.update_xaxes(title_text="x", dtick=1, row=1, col=1)
    fig.update_yaxes(title_text="y", dtick=1, row=1, col=1)
    fig.update_xaxes(title_text="direction m · p means vertical", dtick=1, row=1, col=2)
    fig.update_yaxes(title_text="intercept t", dtick=1, row=1, col=2)
    return fig

line_plot = line_explorer(state)
line_plot.show()

## 4 · Why the reconstruction works

Fix a point $P$. There are $p+1$ lines through it. Any other point $Q$ lies on
exactly one of these lines: if its $x$ coordinate differs, the nonzero difference
has a multiplicative inverse modulo prime $p$, giving one slope; otherwise the
line is vertical.

When we sum the line measurements through $P$, the value $f(P)$ is counted $p+1$
times and every other value once. Therefore

\[
B(P)=(p+1)f(P)+\sum_{Q\ne P}f(Q)=p f(P)+T,
\qquad
\boxed{f(P)=\frac{B(P)-T}{p}}.
\]

For an integer image this division is exact. We inspect its remainder explicitly
before treating `// p` as reconstruction. The same proof works for weighted integer
images when line counts are replaced by `sum(value=F.weight)`.

This is a **finite Radon transform**. Our choice of axes/direction order is explicit;
we do not promise array-order compatibility with another implementation. The standard
prime-grid construction and inverse are documented in
[scikit-image](https://scikit-image.org/docs/stable/api/skimage.transform.html#skimage.transform.frt2).
No scikit-image dependency is needed here.

Primality is essential for this modular-grid argument. Integers modulo a composite
number do not provide the required inverses. A prime-power field would need actual
finite-field arithmetic, not substitution of that number as the modulus.

## 5 · Watch the measurements build another arrangement

For each direction, sum the measured line value onto every incident pixel, then
use that resulting arrangement as a height driver. After all directions, subtract
the common background $T$ and divide by $p$. The final heights reproduce the image.

The target starts with **new occurrences**. Measurements describe the source;
they do not recreate its occurrence identities. The animation moves the target's
own points throughout. Undo restores the captured states and paths.
Grey indicates zero and teal indicates a positive value, using captured endpoint
labels (the color switches endpoints halfway through a move; hover shows both).
Watch the zero pattern emerge when the common background is removed.

In [ ]:
target = (Collection.grid(p, p, values=0).annotate(u=F.i, v=F.j)
          .arrange(F.u, F.v, 0))
stage = target
stages, stage_names = [], []
for direction in range(P + 1):
    contribution = pairs.where(on_line & (F.m == direction)).sum(
        by=(F.u, F.v), value=measured_value)
    amount = contribution.bind(on=(F.u, F.v), key=(F.u, F.v))
    stage = stage.with_values(F.value + amount).move(vector(0, 0, amount))
    stages.append(stage)
    stage_names.append("add vertical family" if direction == P else f"add direction {direction}")

centered = stage.with_values(F.value - total_from_measurements).move(vector(0, 0, -total_from_measurements))
finished = centered.with_values(F.value // p).arrange(F.u, F.v, F.value)
stages.extend((centered, finished))
stage_names.extend(("subtract the common total T", "divide by p · the image appears"))

motion_workspace = Workspace({"target": target}, PARAMETERS)
forward = [motion_workspace.set("target", construction, motion=Motion()) for construction in stages]
finished_state = motion_workspace.state
assert not finished_state.errors, dict(finished_state.errors)
assert keyed_values(finished_state.results["target"], keys=("u", "v")) == keyed_values(state.results["image"], keys=("u", "v"))
assert set(finished_state.results["target"].ids).isdisjoint(state.results["image"].ids)
assert finished_state.results["target"].ids == forward[0].start.results["target"].ids
assert set(forward[-1].start.results["target"].values % P) == {0}
motion_workspace.capture("Line-count fields drive a new arrangement; after removing T and dividing by p it recovers the image values.")
backward = [motion_workspace.undo() for _ in forward]
samples, labels = [], []
for title, transition in zip(stage_names + ["undo · " + s for s in reversed(stage_names)], forward + backward):
    for t in np.linspace(0, 1, 9):
        samples.append(transition.frame("target", float(t)))
        labels.append(f"{title} · {t:.0%}")
np.testing.assert_array_equal(samples[0].positions, samples[-1].positions)
for outward, inward in zip(forward, reversed(backward)):
    np.testing.assert_array_equal(outward.frame("target", .25).positions, inward.frame("target", .75).positions)

reconstruction_motion = animation_figure(samples, labels=labels,
    title="Line measurements lift a second arrangement", duration=65, show_values=False)
# Presentation colors use the current endpoint label, never the original image.
for frame, sample in zip(reconstruction_motion.frames, samples):
    values = sample.values_before if sample.fraction < .5 else sample.values_after
    frame.data[0].marker.color = ["#415266" if value == 0 else TEAL if value > 0 else "#f083a5"
                                  for value in values]
reconstruction_motion.data[0].marker.color = reconstruction_motion.frames[0].data[0].marker.color
reconstruction_motion.update_layout(scene=dict(xaxis_title="x", yaxis_title="y", zaxis_title="accumulated value"))
reconstruction_motion.show()

In [ ]:
comparison_plot = make_subplots(rows=1, cols=3, subplot_titles=["Original values", "Recovered from measurements", "Keyed difference"])
for col, name in enumerate(("image", "recovered", "residual"), 1):
    comparison_plot.add_trace(heatmap(state.results[name], maximum=1), row=1, col=col)
comparison_plot.update_layout(template="plotly_dark", paper_bgcolor=BG, plot_bgcolor=BG, height=390,
                              title="Every value recovered · the explicit difference is zero")
comparison_plot.update_xaxes(title_text="x", dtick=1)
comparison_plot.update_yaxes(title_text="y", dtick=1)
comparison_plot.show()

## 6 · Explain one reconstructed value

Choose `PIXEL`, then inspect the measured lines that contribute to its reconstruction.
Each count's contributors are point/line-pair occurrences. The explicit `(u,v)` binding
identifies the original image points; we keep both levels in the saved explanation.

In [ ]:
PIXEL = (min(2, P - 1), min(2, P - 1))

def explain_pixel(captured, key):
    inspect = Inspection(captured)
    size = captured.parameters["p"]
    back = inspect.measurement(inspect.find("backprojection", key, by=("u", "v")), limit=size + 1)
    assert not back.truncated
    receipts = []
    for term in back.contributors:
        line_read, = term.reads
        measured_line = inspect.measurement(line_read.driver, limit=size)
        assert not measured_line.truncated and term.weight == measured_line.item.value
        contributors = []
        for contribution in measured_line.contributors:
            image_read, = inspect.bindings(contribution.item.ref)
            pixel = inspect.item(image_read.driver)
            contributors.append({"pair_occurrence": contribution.item.ref.occurrence,
                "pixel_key": [pixel.fields[n] for n in ("u", "v")],
                "image_occurrence": pixel.ref.occurrence})
        receipts.append({"line": list(measured_line.key), "count": measured_line.item.value,
                         "binding": line_read.to_dict(), "contributors": contributors})
    total = inspect.item(inspect.find("family_totals", 0)).value
    assert sum(item["count"] for item in receipts) == back.item.value
    assert (back.item.value - total) % size == 0
    return {"pixel": list(key), "lines": receipts, "backprojection": back.item.value,
            "receipt": back.to_dict(), "total_from_measurements": total,
            "recovered": (back.item.value - total) // size}

explanation = explain_pixel(state, PIXEL)
rows = ["| Incident line (m,t) | Measured count | Selected original pixels |", "| --- | ---: | --- |"]
for item in explanation["lines"]:
    rows.append(f"| {tuple(item['line'])} | {item['count']} | "
                + ", ".join(str(tuple(c['pixel_key'])) for c in item["contributors"]) + " |")
display(Markdown("\n".join(rows)))
print(f"At {PIXEL}: ({explanation['backprojection']} - {explanation['total_from_measurements']}) / {P} = {explanation['recovered']}")

## 7 · Edit the source; then deliberately corrupt a measurement

First move the image's selected column. Counts and reconstructed values reevaluate
through their dependencies. Undo restores the captured source and measurements.

Next increase one non-horizontal line count by 1, keeping every key. Its contribution
reaches exactly the $p$ points on that line. The reconstruction numerators there are
no longer divisible by $p$. We display the remainder as a diagnostic and **do not
round it into a claimed reconstruction**.

Finally we delete one complete reconstructed row. A comparison inferred only from
observed keys cannot know that the row should exist, especially where its values were
zero. An independently declared pixel domain reports those missing occurrences
separately from numerical residuals.

In [ ]:
workspace.set_parameters(image_x=(PARAMETERS["image_x"] + 1) % P)
edited_report = check_reconstruction(workspace.state)
workspace.capture("Changing the source column propagates through line counts to exact recovered values.")
workspace.undo()
assert workspace.state is state

corrupted_counts = counts.with_values(F.value + choose((F.m == 1) & (F.t == 0), 1, 0))
corrupted_backprojection = incidence.sum(by=(F.u, F.v), value=corrupted_counts.bind(
    on=(F.m, F.t), key=(F.m, F.t)))
corrupted_total = corrupted_counts.where(F.m == 0).sum().scalar()
bad_numerator = corrupted_backprojection.with_values(F.value - corrupted_total)
bad_remainders = bad_numerator.with_values(F.value % p).arrange(F.u, F.v)
bad_workspace = Workspace({"counts": corrupted_counts, "remainders": bad_remainders}, PARAMETERS)
assert not bad_workspace.state.errors, dict(bad_workspace.state.errors)
bad_field = keyed_values(bad_workspace.state.results["remainders"], keys=("u", "v"))
witnesses = [list(key) for key, value in bad_field.items() if value]
assert set(map(tuple, witnesses)) == {(x, x) for x in range(P)}
assert sum(bad_field.values()) == P
corruption_plot = go.Figure(heatmap(bad_workspace.state.results["remainders"], maximum=1))
corruption_plot.update_layout(template="plotly_dark", paper_bgcolor=BG, plot_bgcolor=BG, height=450,
                              title="One altered line count · p exact-division failures", xaxis_title="x", yaxis_title="y")
corruption_plot.show()
print("Pixels witnessing the altered m=1, t=0 measurement:", witnesses)

# Delete an entire recovered row. A pivot over its observed axes still looks complete.
# The declared domain reports missing keys, separately from values on shared keys.
MISSING_U = min(1, P - 1)
incomplete_recovered = recovered.where(F.u != MISSING_U).select().arrange(F.u, F.v)
missing_workspace = Workspace({"image": image.arrange(F.u, F.v),
                               "recovered": incomplete_recovered}, PARAMETERS)
assert not missing_workspace.state.errors, dict(missing_workspace.state.errors)
full_pixel_domain = tuple((x, y) for x in range(P) for y in range(P))
missing_domain_comparison = compare_keyed_values(
    missing_workspace.state.results["recovered"], missing_workspace.state.results["image"],
    left_keys=("u", "v"), domain=full_pixel_domain)
assert missing_domain_comparison.missing_left == tuple((MISSING_U, y) for y in range(P))
assert not missing_domain_comparison.missing_right
assert not missing_domain_comparison.unexpected_left and not missing_domain_comparison.unexpected_right
assert missing_domain_comparison.values_equal_on_common and not missing_domain_comparison.holds
original_values = keyed_values(missing_workspace.state.results["image"], keys=("u", "v"))
missing_zero_keys = tuple(key for key in missing_domain_comparison.missing_left if original_values[key] == 0)
# Another image can have an all-one deleted row; absence is still a domain failure.
print("Missing zero-valued keys (possibly empty):", missing_zero_keys)
print("Expected-domain comparison finds the absent row:", missing_domain_comparison.missing_left)

## 8 · Keep the investigation

The complete construction graph includes the original measurement process; it is
provenance, not a proof certificate. The written incidence argument establishes the
general reconstruction theorem for a prime field. Assertions validate these finite cases.

This notebook constructs a dense point/line pair domain of size $p^3(p+1)$.
It is intended for small investigations, not a fast imaging backend. Sparse relations
or compiled gathering could later improve execution without changing the mathematical story.

For weighted integer images, use the displayed `weighted_sums` construction as the
measurement source. For missing directions or noisy measurements, the exact inverse
claim needs different assumptions; the remainder check alone is not a complete
consistency test for arbitrary corrupted measurement data.

In [ ]:
workspace.capture("All finite line counts reconstruct the binary image; source and recovered occurrences remain distinct.")
for name, fig in (("ambiguous-projections", ambiguity_plot), ("line-measurements", line_plot),
                  ("reconstruction-and-undo", reconstruction_motion), ("recovered-image", comparison_plot),
                  ("corrupted-measurement", corruption_plot)):
    fig.write_html(OUTPUT / f"{name}.html", include_plotlyjs=True, full_html=True, auto_play=False)
for name, investigation in (("reconstruction", workspace), ("motion", motion_workspace),
                            ("corrupted", bad_workspace), ("missing-domain", missing_workspace)):
    payload = investigation.to_json()
    (OUTPUT / f"{name}-workspace.json").write_text(payload)
    reopened = Workspace.from_json(payload)
    assert not reopened.state.errors
reopened = Workspace.from_json(workspace.to_json())
assert keyed_values(reopened.state.results["recovered"], keys=("u", "v")) == keyed_values(state.results["image"], keys=("u", "v"))
for key in keyed_values(state.results["counts"], keys=("m", "t")):
    assert reopened.state.results["counts"].contributor_ids(key) == state.results["counts"].contributor_ids(key)
(OUTPUT / "cases.json").write_text(json.dumps({"initial": report, "edited": edited_report,
                                              "corrupted_pixels": witnesses,
                                              "missing_domain": missing_domain_comparison.to_dict()}, indent=2))
(OUTPUT / "pixel-explanation.json").write_text(json.dumps(explanation, indent=2))
print("Saved five offline figures, four workspaces, finite checks, and a pixel explanation to", OUTPUT)

An arrangement can represent a picture, a family of measurements, or a reconstruction
field. The same small vocabulary connects them. What changes is the declared domain,
relation, grouping, and correspondence.

Continue exploring with the [future lessons](../docs/lessons/FUTURE_LESSONS.md):
Young diagrams, additive structure, and counting lattice points as shapes grow.